# CFNA 100-hour campaign — turn compute into *defensible* claims

Budget allocation (edit to taste; totals ~95 h with margin):

| Track | Hours | What it buys |
|---|---|---|
| A. VRAM-vs-context scaling bench | 0.5 | the memory-flatness figure vs a dense transformer |
| B. Equal-compute face-off (CFNA 35M vs matched transformer) | 2 x 12 | the honest architecture comparison — win, tie, or lose, it is publishable |
| C. enwik8 train + test bpb | 12 | the first externally comparable number |
| D. base_35m on the XL corpus to its gates + SFT + evals | 45 | the conversational model milestone |
| E. reserve (ladder rungs, re-runs, 90M scouting) | ~12 | slack |

**Claim discipline:** 'breakthrough' is a word the *results* get to use, not
the plan. Tracks A–C produce evidence; the paper's claim table decides what
it earned. Every stage is resume-safe; rerun any cell after a disconnect.
Auto-backup runs during all training.

In [ ]:
# 1) Setup
import os
REPO = "LMMinier/nueronce"; BRANCH = "claude/cfna-neural-core-verify-ldvgn3"
if not os.path.exists("nueronce"):
    !git clone --branch $BRANCH https://github.com/$REPO nueronce
%cd nueronce
!git pull
%pip install -q numpy 'datasets>=2.19,<4' pytest cryptography cffi
import torch; print(torch.__version__, torch.cuda.get_device_name(0))
!python -m pytest tests/test_gpu_amp.py -q
import os
os.environ["PYTHONPATH"] = os.getcwd()  # so `!python scripts/*.py` can import cfna (no build/install)


In [ ]:
# 2) Restore checkpoints + metrics from Drive
from google.colab import drive
from pathlib import Path
import shutil
drive.mount("/content/drive")
SRC = Path("/content/drive/MyDrive/CFNA_checkpoints"); Path("checkpoints").mkdir(exist_ok=True)
for f in SRC.glob("*"):
    if f.is_file(): shutil.copy2(f, Path("checkpoints")/f.name); print("restored:", f.name)

In [ ]:
# 3) TRACK A (30 min): VRAM/time vs context, CFNA base_35m vs dense transformer
#    at matched params. Random weights: memory shape is weight-independent.
!python scripts/bench_context_scaling.py --preset base_35m \
    --baseline-d-model 512 --baseline-layers 10 \
    --contexts 512,1024,2048,4096,8192,16384
import json; print(json.dumps(json.load(open('metrics/context_scaling.json')), indent=1)[:1200])

In [ ]:
# 4) XL corpus (~2.5 GB) for the 35M rung — reuses the forgeloop builder via
#    the corpus-stack pipeline at raised budgets. ~45-60 min once.
!python scripts/dump_corpus_stack.py --out corpus_xl --phase 2 \
    --target-bytes 2500000000 --val-every 20
!du -sh corpus_xl

In [ ]:
# 5) TRACK D: base_35m pretrain sessions (repeat this cell across sessions;
#    LR ladder: 3e-4 -> 1e-4 -> 3e-5 at each plateau). Runs in background
#    with 30-min auto-backup; monitor with cell 5a.
import os, subprocess, threading, time, shutil
from pathlib import Path
LR = "3e-4"; MINUTES = "170"
cmd = ["python", "-u", "scripts/train_checkpoint.py", "--preset", "base_35m",
       "--corpus", "corpus_xl", "--minutes", MINUTES, "--seq", "192",
       "--batch", "32", "--lr", LR, "--amp", "--resume",
       "--out", "checkpoints/cfna_base_35m.pt"]
Path("logs").mkdir(exist_ok=True)
LOG = Path("logs/35m.log")
PROC = subprocess.Popen(cmd, stdout=LOG.open("w"), stderr=subprocess.STDOUT,
                        text=True, start_new_session=True)
def _bk():
    dst = Path("/content/drive/MyDrive/CFNA_checkpoints")
    while PROC.poll() is None:
        time.sleep(1800)
        for n in ["cfna_base_35m.pt", "cfna_base_35m.pt.json"]:
            if Path(f"checkpoints/{n}").exists(): shutil.copy2(f"checkpoints/{n}", dst/n)
        print("auto-backup", time.strftime("%H:%M"), flush=True)
threading.Thread(target=_bk, daemon=True).start()
print("35M training launched, PID", PROC.pid, "| lr", LR)

In [ ]:
# 5a) Monitor (rerun anytime)
import re
print("alive:", PROC.poll() is None)
lines = LOG.read_text(errors="replace").splitlines()
print("\n".join(lines[-8:]))
b = [float(m.group(1)) for l in lines if (m := re.search(r"held-out bpb ([0-9.]+)", l))]
if b: print(f"\nbpb {b[0]:.3f} -> {b[-1]:.3f} (best {min(b):.3f}; 35M gate <= 1.5)")

In [ ]:
# 6) TRACK B: the FACE-OFF. Same corpus, same wall-clock, same seq/batch/AMP,
#    matched params (~34M). Run this for the SAME number of sessions as cell 5.
#    Sizing note: check the printed param count and nudge --d-model/--n-layers
#    until it is within ~5% of 34.4M; record both counts in the report.
!python scripts/train_baseline.py --kind transformer --corpus corpus_xl \
    --d-model 512 --n-layers 10 --n-heads 8 --max-len 512 \
    --minutes 170 --seq 192 --batch 32 --lr 3e-4 --amp --resume \
    --out checkpoints/baseline_tf_34m.pt

In [ ]:
# 6a) Face-off scoreboard: overlay both held-out curves at equal wall-clock
import json
cf = json.load(open("checkpoints/cfna_base_35m.pt.json"))["history"]
tf = json.load(open("checkpoints/baseline_tf_34m.pt.json"))["history"]
def best_at(h, m): return min((r["heldout_bpb"] for r in h if r["minutes"] <= m), default=None)
for mins in (30, 60, 120, 170):
    print(f"@{mins:4d} min  CFNA {best_at(cf, mins)}  vs  TF {best_at(tf, mins)}")
print("\nequal-compute verdict = the smaller bpb at equal minutes. Report either way.")

In [ ]:
# 7) TRACK C: enwik8 — the external yardstick. Build once, train ~4 sessions,
#    then evaluate on the untouched 5MB test split.
!python scripts/eval_enwik8.py --make-corpus corpus_enwik8
!python scripts/train_checkpoint.py --preset base_35m --corpus corpus_enwik8 \
    --minutes 170 --seq 192 --batch 32 --lr 3e-4 --amp --resume \
    --out checkpoints/cfna_enwik8_35m.pt
!python scripts/eval_enwik8.py --eval checkpoints/cfna_enwik8_35m.pt

In [ ]:
# 8) TRACK D finale (after cell 5a shows bpb <= ~1.5): SFT on the 54K set,
#    then the eval battery + transcripts.
!python scripts/build_conversation_sft.py --out-dir data/conversation_sft
!python scripts/train_conversation.py --data data/conversation_sft \
    --preset base_35m --init-from checkpoints/cfna_base_35m.pt --loss response \
    --out-dir checkpoints/conv_35m --minutes 90 --batch 16 --lr 1e-4 --amp
!python scripts/eval_inference_phase2.py --write-suite --checkpoint checkpoints/conv_35m/best.pt || true

In [ ]:
# 8a) Chat with the 35M
from cfna.chat import Conversation, load_checkpoint
m, ck = load_checkpoint("checkpoints/conv_35m/best.pt")
convo = Conversation(m, system="You are CFNA.", temperature=0.2)
for q in ["Hello! Who are you?", "What is two plus two?",
          "What is the capital of France?", "Explain why the sky is blue."]:
    print(f"User: {q}\nCFNA: {convo.say(q)}\n")

In [ ]:
# 9) Bank everything: checkpoints -> Drive, metrics + curves -> GitHub
import shutil
from pathlib import Path
dst = Path("/content/drive/MyDrive/CFNA_checkpoints")
for p in Path("checkpoints").rglob("*.pt"):
    shutil.copy2(p, dst / p.name.replace("best.pt", "conv_35m_best.pt")); print("->", p)
for p in Path("checkpoints").glob("*.json"): shutil.copy2(p, dst / p.name)
from getpass import getpass
token = getpass("GitHub token: ")
!git add metrics/ && git commit -m "100h campaign metrics: scaling bench, face-off, enwik8, 35M" || echo none
!git push https://$token@github.com/$REPO $BRANCH
del token